In [ ]:
# ============================================================
# TASK 3: KNN vs NAIVE BAYES COMPARISON
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import load_digits, fetch_20newsgroups
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
COLORS = ['#2E86AB','#E84855','#3BB273','#F4A261','#8338EC']

# ============================================================
# DIGITS DATASET — KNN ELBOW + NAIVE BAYES COMPARISON
# ============================================================

print("=" * 65)
print("  KNN vs NAIVE BAYES — DIGITS DATASET")
print("=" * 65)

digits = load_digits()
X, y   = digits.data, digits.target
print(f"Shape: {X.shape} | Classes: {len(np.unique(y))} digits (0-9)")

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# ── ELBOW METHOD: K = 1 to 20 ──────────────────────────────
print("\n📊 Running Elbow Method (K=1 to 20)...")
k_range   = range(1, 21)
train_acc = []
val_acc   = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    # 5-fold cross-validation for validation accuracy
    cv_scores = cross_val_score(knn, X_train, y_train, cv=5)
    val_acc.append(cv_scores.mean())
    knn.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, knn.predict(X_train)))

best_k = k_range[np.argmax(val_acc)]
print(f"✅ Best K = {best_k}  |  Val Accuracy = {max(val_acc):.4f}")

# ── TRAIN BEST KNN ──────────────────────────────────────────
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train, y_train)
knn_pred = knn_best.predict(X_test)

knn_acc  = accuracy_score(y_test, knn_pred)
knn_prec = precision_score(y_test, knn_pred, average='weighted')
knn_rec  = recall_score(y_test, knn_pred, average='weighted')
knn_f1   = f1_score(y_test, knn_pred, average='weighted')

print(f"\nKNN (K={best_k}) Results:")
print(f"  Accuracy : {knn_acc:.4f}")
print(f"  Precision: {knn_prec:.4f}")
print(f"  Recall   : {knn_rec:.4f}")
print(f"  F1-Score : {knn_f1:.4f}")

# ── GAUSSIAN NAIVE BAYES ────────────────────────────────────
gnb = GaussianNB()
gnb.fit(X_train, y_train)
gnb_pred = gnb.predict(X_test)

gnb_acc  = accuracy_score(y_test, gnb_pred)
gnb_prec = precision_score(y_test, gnb_pred, average='weighted')
gnb_rec  = recall_score(y_test, gnb_pred, average='weighted')
gnb_f1   = f1_score(y_test, gnb_pred, average='weighted')

print(f"\nGaussian Naive Bayes Results:")
print(f"  Accuracy : {gnb_acc:.4f}")
print(f"  Precision: {gnb_prec:.4f}")
print(f"  Recall   : {gnb_rec:.4f}")
print(f"  F1-Score : {gnb_f1:.4f}")

# ============================================================
# PCA for 2D Decision Boundary Visualization
# ============================================================

pca   = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
X_pca_train, X_pca_test, yp_train, yp_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

# ============================================================
# PLOTS
# ============================================================

fig = plt.figure(figsize=(22, 18))
fig.suptitle('TASK 3 — KNN vs Naive Bayes: Digits Dataset',
             fontsize=16, fontweight='bold')
gs = gridspec.GridSpec(3, 3, hspace=0.45, wspace=0.35)

# Plot 1: Elbow Curve
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(list(k_range), train_acc, COLORS[0], lw=2.5,
         marker='o', markersize=5, label='Train Accuracy')
ax1.plot(list(k_range), val_acc,   COLORS[1], lw=2.5,
         marker='s', markersize=5, label='Validation Accuracy (5-fold CV)')
ax1.axvline(best_k, color=COLORS[2], linestyle='--', lw=2,
            label=f'Best K = {best_k}')
ax1.fill_between(list(k_range), train_acc, val_acc,
                 alpha=0.1, color='purple', label='Overfit Gap')
ax1.set_xlabel('K Value'); ax1.set_ylabel('Accuracy')
ax1.set_title('Elbow Curve: Validation Accuracy vs K', fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.4)
ax1.set_xticks(list(k_range))

# Plot 2: Sample Digits
ax2 = fig.add_subplot(gs[0, 2])
ax2.axis('off')
ax2.set_title('Sample Digits (0-9)', fontweight='bold')
for i in range(10):
    idx = np.where(y == i)[0][0]
    axin = ax2.inset_axes([0.05 + (i%5)*0.19, 0.55 - (i//5)*0.52,
                            0.18, 0.45])
    axin.imshow(digits.images[idx], cmap='Blues')
    axin.set_title(str(i), fontsize=8, pad=1)
    axin.axis('off')

# Plots 3, 4, 5: Decision Boundaries for K=1, K=5, K=15
for ax_idx, k_val in enumerate([1, 5, 15]):
    ax = fig.add_subplot(gs[1, ax_idx])
    knn_db = KNeighborsClassifier(n_neighbors=k_val)
    knn_db.fit(X_pca_train, yp_train)

    xmn, xmx = X_pca[:,0].min()-1, X_pca[:,0].max()+1
    ymn, ymx = X_pca[:,1].min()-1, X_pca[:,1].max()+1
    xx, yy   = np.meshgrid(np.linspace(xmn, xmx, 150),
                             np.linspace(ymn, ymx, 150))
    Z = knn_db.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.2, cmap='tab10')
    scatter = ax.scatter(X_pca[:,0], X_pca[:,1],
                          c=y, cmap='tab10', s=8,
                          alpha=0.6, edgecolors='none')
    db_acc = accuracy_score(yp_test, knn_db.predict(X_pca_test))
    ax.set_title(f'Decision Boundary K={k_val}\n'
                 f'(Acc={db_acc:.3f})', fontweight='bold')
    ax.set_xlabel('PCA Component 1')
    ax.set_ylabel('PCA Component 2')
    ax.grid(True, alpha=0.3)

# Plot 6: KNN vs GNB Comparison Bar
ax6 = fig.add_subplot(gs[2, :2])
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
knn_vals      = [knn_acc, knn_prec, knn_rec, knn_f1]
gnb_vals      = [gnb_acc, gnb_prec, gnb_rec, gnb_f1]

x_pos = np.arange(len(metrics_names))
bw    = 0.35
bars1 = ax6.bar(x_pos - bw/2, knn_vals, bw,
                label=f'KNN (K={best_k})', color=COLORS[0],
                alpha=0.85, edgecolor='white')
bars2 = ax6.bar(x_pos + bw/2, gnb_vals, bw,
                label='Gaussian NB', color=COLORS[1],
                alpha=0.85, edgecolor='white')
for bar, val in [(b,v) for b,v in zip(bars1,knn_vals)] + \
                [(b,v) for b,v in zip(bars2,gnb_vals)]:
    ax6.text(bar.get_x()+bar.get_width()/2,
             bar.get_height()+0.005,
             f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
ax6.set_xticks(x_pos)
ax6.set_xticklabels(metrics_names)
ax6.set_ylim(0.7, 1.05)
ax6.set_title('KNN vs Gaussian NB — All Metrics', fontweight='bold')
ax6.set_ylabel('Score'); ax6.legend(fontsize=10)
ax6.grid(True, alpha=0.3, axis='y')

# Final Summary Table (text)
ax7 = fig.add_subplot(gs[2, 2])
ax7.axis('off')
table_data = [
    ['Model',       'Acc',   'Prec',  'Rec',  'F1'],
    [f'KNN(K={best_k})', f'{knn_acc:.3f}', f'{knn_prec:.3f}',
     f'{knn_rec:.3f}', f'{knn_f1:.3f}'],
    ['Gaussian NB', f'{gnb_acc:.3f}', f'{gnb_prec:.3f}',
     f'{gnb_rec:.3f}', f'{gnb_f1:.3f}'],
]
tbl = ax7.table(cellText=table_data[1:], colLabels=table_data[0],
                cellLoc='center', loc='center',
                bbox=[0, 0.3, 1, 0.6])
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
tbl.auto_set_column_width(col=list(range(5)))
for (r,c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2E86AB')
        cell.set_text_props(color='white', fontweight='bold')
ax7.set_title('Final Comparison Table', fontweight='bold', pad=20)

plt.show()

# ============================================================
# MULTINOMIAL NB — 20 NEWSGROUPS TEXT CLASSIFICATION
# ============================================================

print("\n" + "=" * 65)
print("  MULTINOMIAL NB — 20 NEWSGROUPS TEXT CLASSIFICATION")
print("=" * 65)

# Load only 4 categories for speed
categories = ['sci.med','sci.space','rec.sport.hockey','comp.graphics']
news_train = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers','footers','quotes'))
news_test  = fetch_20newsgroups(subset='test',  categories=categories,
                                 remove=('headers','footers','quotes'))

print(f"Train docs: {len(news_train.data)}")
print(f"Test docs : {len(news_test.data)}")
print(f"Categories: {news_train.target_names}")

# TF-IDF Features
tfidf = TfidfVectorizer(max_features=10000, stop_words='english')
Xt    = tfidf.fit_transform(news_train.data)
Xv    = tfidf.transform(news_test.data)
yt    = news_train.target
yv    = news_test.target

mnb = MultinomialNB(alpha=1.0)   # alpha = Laplace smoothing
mnb.fit(Xt, yt)
mnb_pred = mnb.predict(Xv)

mnb_acc  = accuracy_score(yv, mnb_pred)
mnb_f1   = f1_score(yv, mnb_pred, average='weighted')

print(f"\n✅ MultinomialNB Results:")
print(f"  Accuracy : {mnb_acc:.4f}")
print(f"  F1-Score : {mnb_f1:.4f}")
print(f"\n📋 Detailed Report:\n")
print(classification_report(yv, mnb_pred,
      target_names=news_test.target_names))

# Top words per category
print("📝 Top 8 words per category:")
feature_names = tfidf.get_feature_names_out()
for i, cat in enumerate(news_train.target_names):
    top_idx  = mnb.feature_log_prob_[i].argsort()[-8:][::-1]
    top_words = [feature_names[j] for j in top_idx]
    print(f"  {cat:<20}: {', '.join(top_words)}")

# ── Final Comparison Table ────────────────────────────────────
print("\n" + "=" * 65)
print("  FINAL MODEL COMPARISON TABLE")
print("=" * 65)
print(f"{'Model':<25} {'Accuracy':>10} {'Precision':>10} "
      f"{'Recall':>10} {'F1-Score':>10}")
print("-" * 65)
print(f"{'KNN (K='+str(best_k)+')':<25} {knn_acc:>10.4f} {knn_prec:>10.4f} "
      f"{knn_rec:>10.4f} {knn_f1:>10.4f}")
print(f"{'Gaussian NB':<25} {gnb_acc:>10.4f} {gnb_prec:>10.4f} "
      f"{gnb_rec:>10.4f} {gnb_f1:>10.4f}")
print(f"{'MultinomialNB (Text)':<25} {mnb_acc:>10.4f} {'N/A':>10} "
      f"{'N/A':>10} {mnb_f1:>10.4f}")

print("\n✅ TASK 3 COMPLETE!")